In [ ]:
!pip install google-generativeai

import json
import re
from typing import List, Dict, Any
import google.generativeai as genai
from google.colab import files, drive
from datetime import datetime


print("✅ Setup complete")

In [ ]:
from google.colab import userdata
API_KEY = ""

# Configure Gemini
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-3-flash-preview')

# Test connection
response = model.generate_content("Say 'API is ready'")
print(response.text)

API is ready


In [ ]:
# Upload the cleaned story file (from Agent 1)
print("📤 Upload your story text file:")
uploaded_story = files.upload()
story_file = list(uploaded_story.keys())[0]

with open(story_file, 'r', encoding='utf-8') as f:
    story_text = f.read()

print(f"✅ Story loaded: {len(story_text)} characters")

# Upload the characters JSON file (from Agent 2)
print("\n📤 Upload your characters JSON file:")
uploaded_chars = files.upload()
chars_file = list(uploaded_chars.keys())[0]

with open(chars_file, 'r', encoding='utf-8') as f:
    characters_data = json.load(f)

print(f"✅ Characters loaded: {len(characters_data.get('characters', []))} characters")

📤 Upload your story text file:


Saving cleaned_story.txt to cleaned_story.txt
✅ Story loaded: 11380 characters

📤 Upload your characters JSON file:


Saving characters.json to characters.json
✅ Characters loaded: 3 characters


In [ ]:
def process_dialogue_and_narration(story_text: str, characters_data: Dict, output_file: str = "dialogue_segments_full.json"):
    """
    Process story text into dialogue and narration segments for TTS.
    Output: JSON format with a list of segments.
    """

    print("🎭 Processing dialogue and narration...")

    prompt = """
You are a STRICT dialogue and narration tagging system for audiobook generation.

Your output will be used by a TTS model that takes:
1. text
2. a description of how that text should be spoken

So for each segment, you must produce:
- the exact text
- whether it is narration or dialogue
- who speaks it
- a speaking_description that tells the TTS model how to perform that exact segment

==================================================
MAIN GOAL
==================================================

Read the full story in order and split it into segments.

Each segment must contain:
- index
- type
- speaker
- speaking_description
- text

==================================================
RULES
==================================================

1. KEEP THE TEXT EXACT
- Do NOT rewrite
- Do NOT paraphrase
- Do NOT summarize
- Keep the exact story text for each segment

2. KEEP STORY ORDER
- Segments must stay in the same reading order as the story

3. SEGMENT TYPES
- narration -> normal story text, scene description, action, non-spoken internal exposition
- dialogue -> spoken text by a character

4. SPEAKER
- narration -> always "narrator"
- dialogue -> the character name if clear
- if unclear -> "unknown"

5. SPEAKING_DESCRIPTION (VERY IMPORTANT)
This field is for TTS performance control.

It must describe HOW the segment should sound when spoken.

Write it like a natural performance instruction, for example:
- Calm, steady narration with a slightly mysterious tone.
- Starts angry and forceful, then becomes calmer and more controlled.
- Soft, reassuring delivery with a gentle and warm tone.
- Nervous, uneven speech, speaking quickly as panic builds.
- Slurred, unsteady drunken speech with inconsistent pacing.
- Rough, breathy delivery with occasional coughing.
- Firm and commanding at first, then quieter and more reflective.
- Low, tired voice with slow pacing and emotional heaviness.

Good speaking_description should mention only what is supported by the text:
- emotional progression
- pacing
- intensity
- tone
- vocal quality if clearly implied by the text

It may include styles like:
- calm
- tense
- urgent
- angry
- hesitant
- breathless
- weak
- drunken
- shaky
- commanding
- sarcastic
- grief-stricken
- controlled
- playful
- cold
- pleading
- exhausted
- relieved

Do NOT add unsupported details.
Do NOT invent accent, age, pitch, or gender if the segment text does not support it.
Do NOT make it too short and generic like:
- sad
- angry
- happy

Instead write useful TTS instructions like:
- Starts irritated and clipped, then rises into open anger.
- Quiet, cautious delivery, with controlled breathing and restrained fear.
- Warm, emotional speech with a slight tremble that suggests relief.

6. NARRATION
For narration segments:
- speaker must be "narrator"
- speaking_description should describe how the narration should be read in that moment

Examples:
- Neutral narration with steady pacing and clear delivery.
- Tense narration with controlled pacing and a suspenseful tone.
- Gentle reflective narration with a calm, flowing rhythm.
- Fast, urgent narration with rising intensity.

7. DIALOGUE
For dialogue segments:
- identify the most likely speaker from context
- speaking_description should describe performance for that specific line, not the character's permanent voice profile

8. CONSERVATIVE ATTRIBUTION
- If speaker is not clear, use "unknown"
- Do NOT guess aggressively

==================================================
OUTPUT FORMAT (STRICT JSON ONLY)
==================================================

{{
  "segments": [
    {{
      "index": 1,
      "type": "narration",
      "speaker": "narrator",
      "speaking_description": "Neutral narration with steady pacing and clear delivery.",
      "text": "Exact text from story"
    }},
    {{
      "index": 2,
      "type": "dialogue",
      "speaker": "John",
      "speaking_description": "Starts angry and accusatory, then becomes more controlled and restrained.",
      "text": "Exact spoken text from story"
    }}
  ]
}}

==================================================
EXAMPLES
==================================================

Example 1:
Input:
The room was silent. John took a step forward.
"I trusted you," he said. "But I won't make that mistake again."

Output:
{{
  "segments": [
    {{
      "index": 1,
      "type": "narration",
      "speaker": "narrator",
      "speaking_description": "Tense narration with measured pacing and a quiet sense of pressure.",
      "text": "The room was silent. John took a step forward."
    }},
    {{
      "index": 2,
      "type": "dialogue",
      "speaker": "John",
      "speaking_description": "Starts hurt and angry, then shifts into a firmer, more controlled delivery.",
      "text": "\\"I trusted you,\\" he said. \\"But I won't make that mistake again.\\""
    }}
  ]
}}

--------------------------------------------------

Example 2:
Input:
Mira knelt beside the child.
"It's all right," she said softly. "You're safe with me now."

Output:
{{
  "segments": [
    {{
      "index": 1,
      "type": "narration",
      "speaker": "narrator",
      "speaking_description": "Gentle narration with calm pacing and a soft, reassuring tone.",
      "text": "Mira knelt beside the child."
    }},
    {{
      "index": 2,
      "type": "dialogue",
      "speaker": "Mira",
      "speaking_description": "Soft, careful, and reassuring delivery with warmth and emotional steadiness.",
      "text": "\\"It's all right,\\" she said softly. \\"You're safe with me now.\\""
    }}
  ]
}}

--------------------------------------------------

Example 3:
Input:
"Leave," Arin said.
Then, more quietly, he added, "Please."
Rain battered the windows.

Output:
{{
  "segments": [
    {{
      "index": 1,
      "type": "dialogue",
      "speaker": "Arin",
      "speaking_description": "Begins firm and forceful, then drops into a quieter, more vulnerable tone.",
      "text": "\\"Leave,\\" Arin said.\\nThen, more quietly, he added, \\"Please.\\""
    }},
    {{
      "index": 2,
      "type": "narration",
      "speaker": "narrator",
      "speaking_description": "Dark, atmospheric narration with a heavy and ominous tone.",
      "text": "Rain battered the windows."
    }}
  ]
}}

==================================================
FINAL CHECK
==================================================

Before output:
- Return valid JSON only
- Do NOT wrap output in markdown
- Do NOT include explanation text
- Preserve story order
- Preserve exact text
- Every segment must have:
  - index
  - type
  - speaker
  - speaking_description
  - text
- narration speaker must always be "narrator"

==================================================
INPUT STORY
==================================================

{story_text}
""".format(story_text=story_text)

    response = model.generate_content(prompt)

    if not hasattr(response, "text") or not response.text:
        raise ValueError("Model returned an empty response.")

    json_text = response.text.strip()

    # Remove accidental markdown fences
    json_text = re.sub(r"^```json\s*", "", json_text, flags=re.IGNORECASE)
    json_text = re.sub(r"^```\s*", "", json_text)
    json_text = re.sub(r"\s*```$", "", json_text)

    try:
        data = json.loads(json_text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse model output as JSON.\n\nModel output:\n{json_text}") from e

    if "segments" not in data or not isinstance(data["segments"], list):
        raise ValueError("JSON does not contain a valid 'segments' list.")

    cleaned_segments = []

    for i, seg in enumerate(data["segments"], 1):
        if not isinstance(seg, dict):
            continue

        seg_type = str(seg.get("type", "")).strip().lower()
        speaker = str(seg.get("speaker", "")).strip()
        speaking_description = str(seg.get("speaking_description", "")).strip()
        text = str(seg.get("text", "")).strip()

        if seg_type not in ["narration", "dialogue"]:
            seg_type = "narration"

        if not speaker:
            speaker = "narrator" if seg_type == "narration" else "unknown"

        if seg_type == "narration":
            speaker = "narrator"

        if not speaking_description:
            if seg_type == "narration":
                speaking_description = "Neutral narration with steady pacing and clear delivery."
            else:
                speaking_description = "Clear spoken delivery with natural emotional expression."

        if not text:
            continue

        cleaned_segments.append({
            "index": i,
            "type": seg_type,
            "speaker": speaker,
            "speaking_description": speaking_description,
            "text": text
        })

    data["segments"] = cleaned_segments
    data["total_segments"] = len(cleaned_segments)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"✅ Tagged {data['total_segments']} segments")
    return data

In [ ]:
def process_in_chunks(story_text: str, characters_data: Dict, chunk_size: int = 8000):
    """
    Process long texts in chunks with overlap for context.
    Output: merged JSON file with all segments.
    """
    print(f"📖 Total story length: {len(story_text)} characters")
    print(f"📦 Processing in chunks of {chunk_size} characters...")

    all_segments = []
    overlap = 500

    for i in range(0, len(story_text), chunk_size - overlap):
        chunk = story_text[i:i + chunk_size]
        chunk_num = i // (chunk_size - overlap) + 1

        print(f"\n--- Processing Chunk {chunk_num} ---")

        prompt = """
You are a STRICT dialogue and narration tagging system for audiobook generation.

You must process this story chunk into ordered segments.

Known characters in this story:
{characters}

Use ONLY these names for dialogue speakers if they match the dialogue.
Otherwise use "unknown".

For each segment output:
- type
- speaker
- speaking_description
- text

RULES:
- Keep text exact
- Do not paraphrase
- narration speaker must be "narrator"
- dialogue speaker must be character name if clear, otherwise "unknown"
- speaking_description must be a useful TTS performance instruction, not a one-word label
- keep original order
- return JSON only in this format:

{{
  "segments": [
    {{
      "type": "narration",
      "speaker": "narrator",
      "speaking_description": "Tense narration with measured pacing and quiet suspense.",
      "text": "Exact text"
    }},
    {{
      "type": "dialogue",
      "speaker": "John",
      "speaking_description": "Starts angry and forceful, then becomes more controlled.",
      "text": "Exact spoken text"
    }}
  ]
}}

CHUNK TEXT:
{chunk}
""".format(
    chunk=chunk,
    characters=json.dumps(characters_data["characters"], indent=2)
)

        try:
            response = model.generate_content(prompt)

            if not hasattr(response, "text") or not response.text:
                print(f"⚠️ Empty response for chunk {chunk_num}")
                continue

            json_text = response.text.strip()
            json_text = re.sub(r"^```json\s*", "", json_text, flags=re.IGNORECASE)
            json_text = re.sub(r"^```\s*", "", json_text)
            json_text = re.sub(r"\s*```$", "", json_text)

            chunk_data = json.loads(json_text)

            if "segments" in chunk_data and isinstance(chunk_data["segments"], list):
                for seg in chunk_data["segments"]:
                    if isinstance(seg, dict):
                        seg["chunk"] = chunk_num
                        all_segments.append(seg)

        except Exception as e:
            print(f"⚠️ Error processing chunk {chunk_num}: {e}")
            continue

        print(f"  ✓ Extracted {len([s for s in all_segments if s.get('chunk') == chunk_num])} segments from chunk {chunk_num}")

    cleaned_segments = []

    for i, seg in enumerate(all_segments, 1):
        seg_type = str(seg.get("type", "")).strip().lower()
        speaker = str(seg.get("speaker", "")).strip()
        speaking_description = str(seg.get("speaking_description", "")).strip()
        text = str(seg.get("text", "")).strip()

        if seg_type not in ["narration", "dialogue"]:
            seg_type = "narration"

        if not speaker:
            speaker = "narrator" if seg_type == "narration" else "unknown"

        if seg_type == "narration":
            speaker = "narrator"

        if not speaking_description:
            if seg_type == "narration":
                speaking_description = "Neutral narration with steady pacing and clear delivery."
            else:
                speaking_description = "Clear spoken delivery with natural emotional expression."

        if not text:
            continue

        cleaned_segments.append({
            "index": i,
            "type": seg_type,
            "speaker": speaker,
            "speaking_description": speaking_description,
            "text": text,
            "chunk": seg.get("chunk")
        })

    output_file = "dialogue_segments_full.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump({"segments": cleaned_segments, "total_segments": len(cleaned_segments)}, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Saved {len(cleaned_segments)} total segments to {output_file}")
    return {"segments": cleaned_segments, "total_segments": len(cleaned_segments)}

In [ ]:
# Run full processing
segments_data = process_in_chunks(story_text, characters_data)

print("\n📖 PREVIEW OF PROCESSED SEGMENTS")
print("=" * 60)

for seg in segments_data["segments"][:10]:
    print(f"\n[{seg.get('index')}] Type: {seg.get('type', 'unknown')}")
    print(f"    Speaker: {seg.get('speaker', 'unknown')}")
    print(f"    Text: {seg.get('text', '')[:120]}...")
    print(f"    Speaking Description: {seg.get('speaking_description', '')[:160]}...")
    print("-" * 40)

if len(segments_data["segments"]) > 10:
    print(f"\n... and {len(segments_data['segments']) - 10} more segments")

📖 Total story length: 11380 characters
📦 Processing in chunks of 8000 characters...

--- Processing Chunk 1 ---
  ✓ Extracted 43 segments from chunk 1

--- Processing Chunk 2 ---
  ✓ Extracted 32 segments from chunk 2

✅ Saved 75 total segments to dialogue_segments_full.json

📖 PREVIEW OF PROCESSED SEGMENTS

[1] Type: narration
    Speaker: narrator
    Text: CHAPTER I. DOWN THE RABBIT-HOLE.  

ALICE was beginning to get very tired of sitting by her sister on the bank, and of h...
    Speaking Description: Steady and clear narration, setting a calm introductory tone....
----------------------------------------

[2] Type: dialogue
    Speaker: alice
    Text: “and what is the use of a book,”...
    Speaking Description: A thoughtful, inquisitive internal monologue with a slight touch of boredom....
----------------------------------------

[3] Type: narration
    Speaker: narrator
    Text: thought Alice,...
    Speaking Description: Neutral narration with a short pause....
------------

In [ ]:
# Download the JSON file
files.download("dialogue_segments_full.json")
print("✅ dialogue_segments_full.json downloaded")

# Also create a readable text version
with open("dialogue_script.txt", "w", encoding="utf-8") as f:
    for segment in segments_data["segments"]:
        if segment.get("type") == "narration":
            f.write(f"[NARRATION] {segment.get('text', '')}\n")
            f.write(f"({segment.get('speaking_description', 'Neutral narration')})\n\n")
        else:
            f.write(f"[{segment.get('speaker', 'UNKNOWN')}] {segment.get('text', '')}\n")
            f.write(f"({segment.get('speaking_description', '')})\n\n")

files.download("dialogue_script.txt")
print("✅ dialogue_script.txt downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ dialogue_segments_full.json downloaded


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ dialogue_script.txt downloaded


In [ ]:
# ===== AGENT 3 CHUNKED TEST CELL =====

test_story_text = """
Arjun stepped into the dark forest. The wind whispered through the trees.

"I have a bad feeling about this," Arjun said.

A shadow moved behind him.

"Who's there?" he shouted.

From somewhere deeper in the forest, a calm voice replied,
"You should not have come here."

Arjun tightened his grip on the lantern and looked around.
The trees seemed to lean closer as the silence thickened.

A woman emerged from the mist, her eyes fixed on him.
"If you value your life, turn back now," she said.

But Arjun stood his ground.
"I didn’t come this far to run."
"""

test_characters_data = {
    "characters": {
        "narrator": {
            "name": "Narrator",
            "description": "Neutral voice, adult, clear and steady tone, medium pace, balanced narration style",
            "ref_text": "Arjun stepped into the dark forest. The wind whispered through the trees."
        },
        "arjun": {
            "name": "Arjun",
            "description": "Young adult male voice, medium pitch, medium pace, determined and reactive tone, expressive speaking style",
            "ref_text": "I didn’t come this far to run."
        },
        "woman": {
            "name": "Woman",
            "description": "Adult female voice, calm and stern tone, slow and measured pace, speaks with quiet authority",
            "ref_text": "You should not have come here."
        }
    },
    "total_characters": 3
}

print("🧪 Running chunked Agent 3 test on hardcoded inputs...\n")

segments_data = process_in_chunks(
    test_story_text,
    test_characters_data,
    chunk_size=2000
)

print("\n" + "=" * 60)
print("✅ AGENT 3 CHUNKED OUTPUT")
print("=" * 60)

for seg in segments_data["segments"]:
    print(json.dumps(seg, indent=2, ensure_ascii=False))
    print("-" * 50)

🧪 Running chunked Agent 3 test on hardcoded inputs...

📖 Total story length: 558 characters
📦 Processing in chunks of 2000 characters...

--- Processing Chunk 1 ---
  ✓ Extracted 14 segments from chunk 1

✅ Saved 14 total segments to dialogue_segments_full.json

✅ AGENT 3 CHUNKED OUTPUT
{
  "index": 1,
  "type": "narration",
  "speaker": "narrator",
  "speaking_description": "Neutral and atmospheric narration, steady pace with a sense of mystery.",
  "text": "Arjun stepped into the dark forest. The wind whispered through the trees.",
  "chunk": 1
}
--------------------------------------------------
{
  "index": 2,
  "type": "dialogue",
  "speaker": "arjun",
  "speaking_description": "Young adult male voice, speaking with caution and growing unease.",
  "text": "\"I have a bad feeling about this,\"",
  "chunk": 1
}
--------------------------------------------------
{
  "index": 3,
  "type": "narration",
  "speaker": "narrator",
  "speaking_description": "Neutral narration tag, maintaini

In [ ]:
# ===== DOWNLOAD + SCRIPT EXPORT =====

from google.colab import files

# Check if segments_data exists
if "segments_data" not in globals():
    raise ValueError("segments_data not found. Run the processing cell first.")

# Download JSON file
files.download("dialogue_segments_full.json")
print("✅ dialogue_segments_full.json downloaded")

# Create readable script
with open("dialogue_script.txt", "w", encoding="utf-8") as f:
    for segment in segments_data.get("segments", []):
        if segment.get("type") == "narration":
            f.write(f"[NARRATION] {segment.get('text', '')}\n")
            f.write(f"VOICE: {segment.get('speaking_description', 'Neutral narration')}\n\n")
        else:
            f.write(f"[{segment.get('speaker', 'UNKNOWN').upper()}] {segment.get('text', '')}\n")
            f.write(f"VOICE: {segment.get('speaking_description', '')}\n\n")

# Download text script
files.download("dialogue_script.txt")
print("✅ dialogue_script.txt downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ dialogue_segments_full.json downloaded


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ dialogue_script.txt downloaded
